Day 1：感知器（Perceptron）
## 1. 今天学习的内容

今天学习了《零基础入门深度学习》的第一部分：感知器。

主要学习内容：

感知器的基本概念
输入、权重、偏置
激活函数
感知器的计算过程
感知器的训练方法
使用感知器实现 AND 逻辑运算

## 2. 感知器是什么

感知器可以看成一个非常简单的神经元。

它接收多个输入：

x1, x2, x3 ...

每个输入都有对应的权重：

w1, w2, w3 ...

然后计算：

w1*x1 + w2*x2 + ... + b

其中：

x：输入
w：权重
b：偏置

最后把计算结果交给激活函数，得到最终输出。(这里的激活函数看上去阶跃函数用的比较多(单层感知器)，主要是二分类，个人感觉换激活函数差别很大比如说RELU)
可以表示为：
y = f(w · x + b)

## 3. 我对权重和偏置的理解
权重 w

权重表示某个输入对最终结果的重要程度。

权重越大，这个输入对结果的影响通常越大。

偏置 b

偏置可以理解为调整判断标准的一个参数。

模型训练过程中不仅会修改权重，也会修改偏置。
(网上给的解释比较多的是平移决策边界，不一定经过原点)
文档中举例用的AND感知器还有OR感知器，且异或运算无法实现，无法用直线去分类

## 4. 今天实现的代码
实现了：
Perceptron 类
AND 感知器
训练过程
预测过程

运行结果：

0 AND 0 => 0
0 AND 1 => 0
1 AND 0 => 0
1 AND 1 => 1

结果符合 AND 运算的预期。

## 5. 今天理解的训练过程

训练感知器的大致流程：

输入训练数据-根据当前权重进行预测-比较预测结果和正确答案-计算误差-修改权重和偏置-继续训练

## 6. 今天遇到的问题

目前还需要继续理解：

为什么权重更新公式可以让结果越来越准确？
learning rate（学习率）具体有什么作用？
权重和偏置最终为什么会收敛到某些值？

这些问题后面学习梯度下降和神经网络时继续理解。

In [8]:
from functools import reduce

class VectorOp:
    @staticmethod
    def dot(x,y):
        return reduce(
            lambda a, b: a + b,
            VectorOp.element_multiply(x,y),
            0.0
        )
    @staticmethod
    def element_multiply(x,y):
        return list(
            map(lambda xy: xy[0] * xy[1], zip(x,y))
        )
    @staticmethod
    def element_add(x, y):
        return list(
            map(lambda xy: xy[0] + xy[1], zip(x,y))
        )
    @staticmethod
    def scalar_multiply(v, s):
        return list(map(lambda e: e * s, v))
class Perception:
    def __init__(self, input_num, activator):
        self.activator = activator
        self.weights=[0.0]*input_num
        self.bias=0.0

    def predict(self, input_vec):
        weighted_sum=VectorOp.dot(
            input_vec,
            self.weights  
        )+self.bias
        return self.activator(weighted_sum)
    def f(x):
        return 1 if x>0 else 0
    def train(self, input_vecs, labels, iteration, rate):
        for i in range(iteration):
            self._one_iteration(input_vecs, labels, rate)
    def _one_iteration(self, input_vecs, labels, rate):
        samples = zip(input_vecs, labels)
        for input_vec, label in samples:
            output = self.predict(input_vec)
            self._update_weights(
                input_vec,
                output,
                label,
                rate
            )
    def _update_weights(self, input_vec, output, label, rate):
        delta=label-output
        self.weights=VectorOp.element_add(
            self.weights,
            VectorOp.scalar_multiply(
                input_vec,
                rate*delta
            )
        )
        self.bias += rate * delta
def f(x):
    return 1 if x > 0 else 0
def get_training_dataset():
    input_vecs = [
        [1, 1],
        [0, 0],
        [1, 0],
        [0, 1]
    ]

    labels = [
        1,
        0,
        0,
        0
    ]

    return input_vecs, labels
def train_and_perception():
    p = Perception(2, f)

    input_vecs, labels = get_training_dataset()

    p.train(
        input_vecs,
        labels,
        iteration=10,
        rate=0.1
    )

    return p
and_perception = train_and_perception()

print("1 AND 1 =", and_perception.predict([1, 1]))
print("0 AND 0 =", and_perception.predict([0, 0]))
print("1 AND 0 =", and_perception.predict([1, 0]))
print("0 AND 1 =", and_perception.predict([0, 1]))
print("weights =", and_perception.weights)
print("bias =", and_perception.bias)


1 AND 1 = 1
0 AND 0 = 0
1 AND 0 = 0
0 AND 1 = 0
weights = [0.1, 0.2]
bias = -0.2


In [9]:
class Perceptron(object):
    def __init__(self, input_num, activator):
        self.activator = activator
        self.weights = [0.0 for _ in range(input_num)]
        self.bias = 0.0

    def __str__(self):
        return 'weights\t:%s\nbias\t:%f\n' % (self.weights, self.bias)

    def predict(self, input_vec):
        return self.activator(
            reduce(
                lambda a, b: a + b,
                map(
                    lambda pair: pair[0] * pair[1],
                    zip(input_vec, self.weights)
                ),
                0.0
            ) + self.bias
        )

    def train(self, input_vecs, labels, iteration, rate):
        for i in range(iteration):
            self._one_iteration(input_vecs, labels, rate)

    def _one_iteration(self, input_vecs, labels, rate):
        samples = zip(input_vecs, labels)
        for input_vec, label in samples:
            output = self.predict(input_vec)
            self._update_weights(input_vec, output, label, rate)

    def _update_weights(self, input_vec, output, label, rate):
        delta = label - output
        self.weights = [
            w + rate * delta * x
            for x, w in zip(input_vec, self.weights)
        ]
        self.bias += rate * delta


def f(x):
    return 1 if x > 0 else 0


def get_training_dataset():
    input_vecs = [[1, 1], [0, 0], [1, 0], [0, 1]]
    labels = [1, 0, 0, 0]
    return input_vecs, labels


def train_and_perceptron():
    p = Perceptron(2, f)
    input_vecs, labels = get_training_dataset()
    p.train(input_vecs, labels, 10, 0.1)
    return p


and_perceptron = train_and_perceptron()
print(and_perceptron)
print('1 and 1 = %d' % and_perceptron.predict([1, 1]))
print('0 and 0 = %d' % and_perceptron.predict([0, 0]))
print('1 and 0 = %d' % and_perceptron.predict([1, 0]))
print('0 and 1 = %d' % and_perceptron.predict([0, 1]))
    

        

weights	:[0.1, 0.2]
bias	:-0.200000

1 and 1 = 1
0 and 0 = 0
1 and 0 = 0
0 and 1 = 0
